### Block 1 - Set up the API connection
Imports the necessary libraries and loads the Last.fm API key securely from a .env file. Since I'm a fan of Ariana for almost 5 years, I choose her as the artist i want to analyze.

In [70]:
import os, re, time, requests
import pandas as pd
from dotenv import load_dotenv

# Load the API key from .env so it never appears in the code
load_dotenv()
API_KEY = os.getenv("LASTFM_API_KEY")
BASE_URL = "https://ws.audioscrobbler.com/2.0/"
ARTIST = "Ariana Grande"

### Block 2 - Create an API helper and select albums
Defines a reusable function to retrieve data from Last.fm and handle potential errors. I looked through Ariana Grande’s top albums on Last.fm and selected five, including some of my personal favorites. These albums span different stages of her career, each with its own distinct theme.

In [71]:
def lastfm(method, **params):
    """Call a Last.fm API method and return the JSON response."""
    params.update(method=method, api_key=API_KEY, format="json")
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    if "error" in data:  # Last.fm reports some errors inside the JSON
        raise ValueError(f"Last.fm error {data['error']}: {data['message']}")
    return data

# Test: list the artist's top albums on Last.fm
top = lastfm("artist.gettopalbums", artist=ARTIST, limit=30)
[a["name"] for a in top["topalbums"]["album"]]

ALBUMS = ["eternal sunshine", "Positions", "Yours Truly", "Sweetener", "thank u, next"]

### Block 3 - Collect album track information
Goes through each selected album and collects its songs, track numbers, and durations, and organizes the info into a data frame.

In [72]:
# Create a list to collect one row of data per track.
rows = []

# Request track information for each album in ALBUMS.
for album in ALBUMS:
    info = lastfm("album.getinfo", artist=ARTIST, album=album)["album"]

    # Get the album's tracks; use an empty list if track data is missing.
    tracks = info.get("tracks", {}).get("track", [])

    # Last.fm may return a single track as a dictionary instead of a list.
    # Wrap it in a list so the loop below works in either case.
    if isinstance(tracks, dict):
        tracks = [tracks]

    # Add each track's details as a row.
    for t in tracks:
        rows.append({
            "album": info["name"],
            "song_name": t["name"],
            "track_number": int(t["@attr"]["rank"]),
            "duration_sec": t.get("duration"),  # Duration may be missing.
        })

    # Pause briefly between requests to avoid calling the API too rapidly.
    time.sleep(0.25)

# Turn the collected rows into a pandas DataFrame.
lastfm_df = pd.DataFrame(rows)

# Convert durations to numbers; values that are missing or invalid become NaN.
lastfm_df["duration_sec"] = pd.to_numeric(
    lastfm_df["duration_sec"], errors="coerce"
)

# Print the DataFrame's dimensions: (number of rows, number of columns).
print(lastfm_df.shape)

# Display the first 20 rows.
lastfm_df.head(20)

(66, 4)


,album,song_name,track_number,duration_sec
0,eternal sunshine,intro (end of the world),1,92.0
1,eternal sunshine,bye,2,164.0
2,eternal sunshine,don't wanna break up again,3,174.0
3,eternal sunshine,Saturn Returns Interlude,4,42.0
4,eternal sunshine,eternal sunshine,5,210.0
5,eternal sunshine,supernatural,6,163.0
6,eternal sunshine,true story,7,163.0
7,eternal sunshine,the boy is mine,8,112.0
8,eternal sunshine,"yes, and?",9,214.0
9,eternal sunshine,we can't be friends (wait for your love),10,227.0


### Block 4 - Fix incorrect song names
Considering that two song titles (thank you, next; You'll Never Know (Live)) that are listed incorrectly on Last.fm, I correct them so I can retrieve information for the right songs in the next step. 

In [73]:
# Last.fm's album data lists this Yours Truly track as "(Live)",
# but the album has the studio version, so rename it before looking up its details
# Correct two track names that Last.fm's album data has wrong
name_fixes = {
    "You'll Never Know (Live)": "You'll Never Know",   # album has the studio version, not live
    "thank you, next": "thank u, next",                # official title uses "u"
}
fix = lastfm_df["song_name"].isin(name_fixes.keys())
lastfm_df.loc[fix, "song_name"] = lastfm_df.loc[fix, "song_name"].replace(name_fixes)
lastfm_df.loc[lastfm_df["song_name"] == "You'll Never Know", "duration_sec"] = None   # refill studio duration
lastfm_df[fix][["album", "track_number", "song_name"]]

,album,track_number,song_name
35,Yours Truly,9,You'll Never Know
64,"thank u, next",11,"thank u, next"


### Block 5 - Collect song popularity data
I choose to collect the number of listeners, play counts, and available durations for each song. It also handles failed requests and fills in missing durations when possible.

In [74]:
listeners, playcounts, durations = [], [], []
for _, row in lastfm_df.iterrows():
    try:
        tr = lastfm("track.getinfo", artist=ARTIST, track=row["song_name"])["track"]
        listeners.append(int(tr.get("listeners", 0)))
        playcounts.append(int(tr.get("playcount", 0)))
        ms = int(tr.get("duration", 0) or 0)
        durations.append(ms / 1000 if ms > 0 else None)
    except Exception as e:      # keep going if one song fails
        print("Failed:", row["song_name"], "-", e)
        listeners.append(None); playcounts.append(None); durations.append(None)
    time.sleep(0.25)

lastfm_df["listeners"] = listeners
lastfm_df["playcount"] = playcounts
# Use track.getinfo's duration only where album.getinfo had none (or 0)
missing = lastfm_df["duration_sec"].isna() | (lastfm_df["duration_sec"] == 0)
lastfm_df.loc[missing, "duration_sec"] = pd.Series(durations)[missing]

lastfm_df.isna().sum()

album           0
song_name       0
track_number    0
duration_sec    5
listeners       0
playcount       0
dtype: int64

### Block 6 - Update Song Popularity Data
Matches each song with its main Last.fm entry, excluding alternative versions. This ensures more accurate listener and play counts for popularity comparisons.

In [75]:

# Get the artist's top tracks (main Last.fm entries), several pages to cover album tracks
top_tracks = []
for page in [1, 2, 3]:
    top_tracks += lastfm("artist.gettoptracks", artist=ARTIST, limit=500, page=page)["toptracks"]["track"]
    time.sleep(0.25)

def lfm_key(s):
    """Simplify a title so different spellings of the same song match."""
    s = s.lower().replace("’", "'")
    s = re.sub(r"\s*[\(\[].*?[\)\]]", "", s)   # drop "(feat. ...)", "[Remix]" etc.
    s = re.sub(r"\s+-\s+.*$", "", s)           # drop " - remix", " - live" etc.
    return re.sub(r"[^\w]", "", s)

# Skip other recordings of a song, keep the entry with the most listeners per title
SKIP = ("remix", "live", "acoustic", "instrumental", "demo", "sped up", "slowed")
best = {}
for t in top_tracks:
    if any(word in t["name"].lower() for word in SKIP):
        continue
    k, n = lfm_key(t["name"]), int(t["listeners"])
    if k not in best or n > int(best[k]["listeners"]):
        best[k] = t

# Replace each song's numbers with its main entry, and record what changed
changes = []
for i, row in lastfm_df.iterrows():
    t = best.get(lfm_key(row["song_name"]))
    if t is None:
        changes.append((row["song_name"], row["listeners"], None, "not in top tracks, kept original"))
        continue
    new = int(t["listeners"])
    if new != row["listeners"]:
        changes.append((row["song_name"], row["listeners"], new, t["name"]))
    lastfm_df.at[i, "listeners"] = new
    lastfm_df.at[i, "playcount"] = int(t["playcount"])

pd.DataFrame(changes, columns=["song", "old_listeners", "new_listeners", "main_entry_used"])

,song,old_listeners,new_listeners,main_entry_used
0,ordinary things,8169,508218,ordinary things (feat. Nonna)
1,34+35,1964,1133674,34+35
2,MOTIVE,18079,948617,motive (with Doja Cat)
3,off the table,11931,454044,off the table (with The Weeknd)
4,safety net,11451,567449,safety net (feat. Ty Dolla $ign)
5,Almost Is Never Enough,11333,412148,Almost Is Never Enough
6,blazed,12726,339743,blazed (feat. Pharrell Williams)
7,the light is coming,21595,432379,the light is coming (feat. Nicki Minaj)
8,borderline,12231,405986,borderline (feat. Missy Elliott)


### Block 7 - Identify missing durations
Checks which songs still have missing duration values.

In [76]:
lastfm_df[lastfm_df["duration_sec"].isna()][["album", "track_number", "song_name"]]

,album,track_number,song_name
17,Positions,5,off the table
39,Sweetener,1,raindrops (an angel cried)
41,Sweetener,3,the light is coming
50,Sweetener,12,better off
53,Sweetener,15,get well soon


### Block 8 - Fill in missing durations
Manually fills in the remaining missing song durations using information from Apple Music.

In [77]:
# Durations missing from Last.fm, filled in from Apple Music (minutes * 60 + seconds)
manual_durations = {
    "off the table": 3 * 60 + 59,
    "raindrops (an angel cried)": 37,
    "the light is coming": 3 * 60 + 48,
    "better off": 2 * 60 + 51,
    "get well soon": 5 * 60 + 22,
}
lastfm_df["duration_sec"] = lastfm_df["duration_sec"].fillna(
    lastfm_df["song_name"].map(manual_durations)
)
print(lastfm_df["duration_sec"].isna().sum())

0


### Block 9 - Save the track dataset
Exports the cleaned dataframe to a CSV file for future analysis without calling the API again.

In [78]:
lastfm_df.to_csv("../data/lastfm_tracks.csv", index=False)
lastfm_df.head()

,album,song_name,track_number,duration_sec,listeners,playcount
0,eternal sunshine,intro (end of the world),1,92.0,927701,20102803
1,eternal sunshine,bye,2,164.0,869065,17990162
2,eternal sunshine,don't wanna break up again,3,174.0,683405,14602788
3,eternal sunshine,Saturn Returns Interlude,4,42.0,545335,9299475
4,eternal sunshine,eternal sunshine,5,210.0,772584,19106948
